# Análisis del Agente Connect-4: MCTS/UCT
**Cristian Castañeda — Fundamentos de Inteligencia Artificial 2026.1**

Este notebook valida y analiza el agente basado en **Monte Carlo Tree Search (MCTS/UCT)** para el torneo de Connect-4.
El estudio cubre:
1. Rendimiento base vs jugador aleatorio (ambos colores)
2. Efecto del número de simulaciones *(variable numérica de configuración)*
3. Comparación de dos versiones: **V1** (MCTS + táctica inmediata) vs **V2** (MCTS puro)
4. Auto-desempeño *(self-play)*
5. Efecto de la constante de exploración **C** de UCB1
6. Análisis de cuellos de botella y propuesta de mejoras

---
## 0. Configuración del entorno

In [ ]:
import sys, os

def find_tournament_root():
    path = os.getcwd()
    for _ in range(7):
        if os.path.isdir(os.path.join(path, 'connect4')):
            return path
        path = os.path.dirname(path)
    raise RuntimeError('No se encontro el directorio raiz (carpeta connect4/)')

ROOT = find_tournament_root()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print('Directorio de trabajo:', ROOT)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
})

from connect4.connect_state import ConnectState
from connect4.policy import Policy
from groups.Cristian.policy import Cristian, _MCTSNode

print('Importaciones exitosas')

---
## Infraestructura de experimentación

In [ ]:
class RandomAgent(Policy):
    """Agente de referencia: elige columna valida al azar."""
    def mount(self): pass
    def act(self, s):
        cols = [c for c in range(7) if s[0, c] == 0]
        return int(np.random.default_rng().choice(cols))


def play_game(red_policy, yellow_policy):
    """Juega una partida. Retorna ganador: -1=Rojo, 1=Amarillo, 0=Empate."""
    rng = np.random.default_rng()
    state = ConnectState()
    while not state.is_final():
        policy = red_policy if state.player == -1 else yellow_policy
        col = policy.act(state.board)
        if not state.is_applicable(col):
            col = int(rng.choice(state.get_free_cols()))
        state = state.transition(col)
    return state.get_winner()


def evaluate(agent_cls, opp_cls, n_games, as_color, agent_kw=None, opp_kw=None):
    """
    Evalua agent_cls vs opp_cls.
    as_color: -1 -> agente juega Rojo, 1 -> juega Amarillo.
    Retorna dict con wins, draws, losses, wr (win-rate), lr (loss-rate).
    """
    agent_kw = agent_kw or {}
    opp_kw   = opp_kw or {}
    wins = draws = losses = 0
    for _ in range(n_games):
        agent = agent_cls(**agent_kw); agent.mount()
        opp   = opp_cls(**opp_kw);    opp.mount()
        result = play_game(agent, opp) if as_color == -1 else play_game(opp, agent)
        if result == as_color:  wins   += 1
        elif result == 0:       draws  += 1
        else:                   losses += 1
    return {'wins': wins, 'draws': draws, 'losses': losses,
            'wr': wins / n_games, 'lr': losses / n_games}


# Numero de partidas por configuracion.
# Aumentar a 100 para mayor precision estadistica (tarda ~2x mas).
N = 50
print('N =', N, 'partidas por configuracion')

---
## Experimento 1: Rendimiento base vs Jugador Aleatorio

El agente juega **N partidas como Rojo** (primer jugador, `-1`) y **N partidas como Amarillo** (segundo jugador, `+1`) contra el jugador aleatorio.

**Requisitos mínimos de la rúbrica:** nunca perder Y ganar ≥ 50 % en ambos colores.

In [ ]:
print('Experimento 1 — Cristian (500 sims) vs Aleatorio...\n')

res_red = evaluate(Cristian, RandomAgent, N, -1, agent_kw={'simulations': 500})
res_yel = evaluate(Cristian, RandomAgent, N,  1, agent_kw={'simulations': 500})

for color, res in [('Rojo (-1)',    res_red),
                   ('Amarillo (+1)', res_yel)]:
    wr  = res['wr']  * 100
    dr  = res['draws'] / N * 100
    lr  = res['lr']  * 100
    print(f'Como {color}:')
    print(f'  Victorias : {res["wins"]:>3} / {N}  ({wr:.1f}%)')
    print(f'  Empates   : {res["draws"]:>3} / {N}  ({dr:.1f}%)')
    print(f'  Derrotas  : {res["losses"]:>3} / {N}  ({lr:.1f}%)\n')

ok_wr   = res_red['wr'] >= 0.5 and res_yel['wr'] >= 0.5
ok_loss = res_red['losses'] == 0 and res_yel['losses'] == 0
print('Requisito >=50% victorias :', 'CUMPLIDO' if ok_wr   else 'NO cumplido')
print('Requisito 0 derrotas      :', 'CUMPLIDO' if ok_loss else 'NO cumplido')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

palette = {'Victorias': '#2ecc71', 'Empates': '#f39c12', 'Derrotas': '#e74c3c'}
cats    = ['Victorias', 'Empates', 'Derrotas']

for ax, (lbl, res) in zip(axes, [('Rojo  (mueve primero)', res_red),
                                   ('Amarillo  (mueve segundo)', res_yel)]):
    vals = [res['wins'], res['draws'], res['losses']]
    bars = ax.bar(cats, vals, color=[palette[c] for c in cats],
                  edgecolor='white', linewidth=1.8, width=0.5)
    ax.set_title('Cristian como ' + lbl, fontweight='bold')
    ax.set_ylabel('Partidas')
    ax.set_ylim(0, N * 1.25)
    ax.axhline(N * 0.5, color='#7f8c8d', ls='--', lw=1.2, label='50 %')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.6, str(val),
                ha='center', va='bottom', fontweight='bold')
    ax.legend(fontsize=9)

fig.suptitle('Exp. 1 — Cristian (500 sims) vs Agente Aleatorio  |  N=' + str(N) + ' partidas/color',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('groups/Cristian/exp1_baseline.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafica guardada: exp1_baseline.png')

---
## Experimento 2: Efecto del número de simulaciones

Se varía el parámetro `simulations` ∈ {25, 50, 100, 200, 500} y se mide la tasa de victoria.

Esta es la **variable numérica de configuración** requerida por la rúbrica.
Más simulaciones → árbol MCTS más profundo → mejores decisiones, pero mayor tiempo de cómputo.

In [ ]:
sim_values  = [25, 50, 100, 200, 500]
wr_red_sim  = []
wr_yel_sim  = []

print('Experimento 2 — Barrido de simulaciones...')
for sims in sim_values:
    r = evaluate(Cristian, RandomAgent, N, -1, agent_kw={'simulations': sims})
    y = evaluate(Cristian, RandomAgent, N,  1, agent_kw={'simulations': sims})
    wr_r = round(r['wr'] * 100, 1)
    wr_y = round(y['wr'] * 100, 1)
    wr_red_sim.append(r['wr'])
    wr_yel_sim.append(y['wr'])
    print('  sims=' + str(sims).rjust(4) + ': Rojo=' + str(wr_r) + '%  Amarillo=' + str(wr_y) + '%')

print('\nExperimento 2 completado.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(sim_values, [v * 100 for v in wr_red_sim],
        'o-', color='#e74c3c', lw=2.5, ms=9, label='Como Rojo')
ax.plot(sim_values, [v * 100 for v in wr_yel_sim],
        's--', color='#e67e22', lw=2.5, ms=9, label='Como Amarillo')
ax.axhline(50, color='#7f8c8d', ls=':', lw=1.5, label='50 % minimo')
ax.fill_between(sim_values,
                [v * 100 for v in wr_red_sim],
                [v * 100 for v in wr_yel_sim],
                alpha=0.08, color='#3498db')

ax.set_xlabel('Numero de simulaciones MCTS')
ax.set_ylabel('Tasa de victoria (%)')
ax.set_title('Exp. 2 — Win rate vs Simulaciones  (vs Agente Aleatorio)', fontweight='bold')
ax.set_xscale('log')
ax.set_xticks(sim_values)
ax.get_xaxis().set_major_formatter(mticker.ScalarFormatter())
ax.set_ylim(0, 110)
ax.legend()
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig('groups/Cristian/exp2_simulations.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafica guardada: exp2_simulations.png')

---
## Experimento 3: Dos versiones — Con y Sin acción táctica inmediata

El módulo `_immediate_action` detecta en **O(cols)** si existe un movimiento ganador propio o un bloqueo urgente,
*antes* de gastar el presupuesto de simulaciones MCTS.

| Versión | Descripción |
|---------|-------------|
| **V1 — Completa** | MCTS + acción táctica inmediata *(agente oficial)* |
| **V2 — Solo MCTS** | MCTS puro, sin módulo táctico |

In [ ]:
class CristianNoTactic(Cristian):
    """Version 2: MCTS puro, sin accion tactica inmediata."""
    def act(self, s):
        me   = self._infer_player(s)
        root = _MCTSNode(s.copy(), me)
        for _ in range(self.simulations):
            leaf   = self._select(root)
            child  = self._expand(leaf)
            reward = self._simulate(child.board, child.to_move, me)
            self._backpropagate(child, reward)
        if not root.children:
            return self._valid_cols(s)[0]
        return max(root.children, key=lambda ch: ch.visits).action

print('CristianNoTactic definido (V2: solo MCTS)')

In [ ]:
print('Experimento 3 — V1 vs V2 contra Aleatorio (200 sims)...')

v1r = evaluate(Cristian,         RandomAgent, N, -1, agent_kw={'simulations': 200})
v1y = evaluate(Cristian,         RandomAgent, N,  1, agent_kw={'simulations': 200})
v2r = evaluate(CristianNoTactic, RandomAgent, N, -1, agent_kw={'simulations': 200})
v2y = evaluate(CristianNoTactic, RandomAgent, N,  1, agent_kw={'simulations': 200})

for lbl, r, y in [('V1 (MCTS + Tactica)', v1r, v1y),
                   ('V2 (Solo MCTS)',       v2r, v2y)]:
    wr_r = round(r['wr'] * 100, 1)
    lr_r = round(r['lr'] * 100, 1)
    wr_y = round(y['wr'] * 100, 1)
    lr_y = round(y['lr'] * 100, 1)
    print('\n' + lbl + ':')
    print('  Rojo     -> Victoria: ' + str(wr_r) + '%  Derrota: ' + str(lr_r) + '%')
    print('  Amarillo -> Victoria: ' + str(wr_y) + '%  Derrota: ' + str(lr_y) + '%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, color_lbl, res_v1, res_v2, as_c in [
        (axes[0], 'Rojo  (primer jugador)',    v1r, v2r, 'Rojo'),
        (axes[1], 'Amarillo  (segundo jugador)', v1y, v2y, 'Amarillo')]:

    cats_v  = ['Victorias', 'Empates', 'Derrotas']
    v1_vals = [res_v1['wins'], res_v1['draws'], res_v1['losses']]
    v2_vals = [res_v2['wins'], res_v2['draws'], res_v2['losses']]

    x    = np.arange(len(cats_v))
    w    = 0.35
    bars1 = ax.bar(x - w/2, v1_vals, width=w, label='V1 (+Tactica)',
                   color='#27ae60', edgecolor='white', lw=1.5)
    bars2 = ax.bar(x + w/2, v2_vals, width=w, label='V2 (Solo MCTS)',
                   color='#2980b9', edgecolor='white', lw=1.5)

    for bar, val in list(zip(bars1, v1_vals)) + list(zip(bars2, v2_vals)):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.4, str(val),
                ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(cats_v)
    ax.set_title('Como ' + color_lbl, fontweight='bold')
    ax.set_ylabel('Partidas')
    ax.set_ylim(0, N * 1.3)
    ax.axhline(N * 0.5, color='#7f8c8d', ls='--', lw=1, label='50 %')
    ax.legend(fontsize=9)

fig.suptitle('Exp. 3 — V1 (MCTS+Tactica) vs V2 (Solo MCTS)  |  200 sims, N=' + str(N),
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('groups/Cristian/exp3_versions.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafica guardada: exp3_versions.png')

---
## Experimento 4: Auto-desempeño (Self-play)

El agente juega contra sí mismo (`Cristian` vs `Cristian`, mismas simulaciones).
En Connect-4 el primer jugador (Rojo) tiene **ventaja estructural** demostrada.
Resultados asimétricos confirman que el agente sí explota esa ventaja en vez de jugar simétricamente.

In [ ]:
print('Experimento 4 — Self-play (200 sims)...')

sp_red = sp_yel = sp_draw = 0
for _ in range(N):
    a1 = Cristian(simulations=200); a1.mount()
    a2 = Cristian(simulations=200); a2.mount()
    r  = play_game(a1, a2)
    if   r == -1: sp_red  += 1
    elif r ==  1: sp_yel  += 1
    else:         sp_draw += 1

print('Rojo gana     :', sp_red,  '(' + str(round(sp_red / N * 100, 1)) + '%)')
print('Amarillo gana :', sp_yel,  '(' + str(round(sp_yel / N * 100, 1)) + '%)')
print('Empates       :', sp_draw, '(' + str(round(sp_draw / N * 100, 1)) + '%)')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

lbls_sp = ['Rojo gana\n(primer jugador)', 'Empate', 'Amarillo gana\n(segundo jugador)']
vals_sp = [sp_red, sp_draw, sp_yel]
clrs_sp = ['#e74c3c', '#95a5a6', '#f1c40f']

bars = ax.bar(lbls_sp, vals_sp, color=clrs_sp, edgecolor='white', lw=1.8, width=0.45)
for bar, val in zip(bars, vals_sp):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.3, str(val),
            ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.axhline(N / 2, color='#7f8c8d', ls='--', lw=1.2,
           label='50 % (' + str(N // 2) + ' partidas)')
ax.set_ylabel('Numero de partidas')
ax.set_ylim(0, N * 1.3)
ax.set_title('Exp. 4 — Cristian vs Cristian (Self-play)\n200 sims, N=' + str(N),
             fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('groups/Cristian/exp4_selfplay.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafica guardada: exp4_selfplay.png')

---
## Experimento 5: Efecto de la constante de exploración C

UCB1 = $\frac{w_i}{n_i} + C \sqrt{\frac{\ln N}{n_i}}$

- **C → 0**: explotación pura (el árbol sigue siempre la rama con mayor win-rate)
- **C → ∞**: exploración pura (equivale a aleatorio)
- **C = √2 ≈ 1.414**: valor teórico óptimo para juegos con recompensas ∈ [0,1]

Se estudia si la elección de C afecta el rendimiento contra el aleatorio con 200 simulaciones.

In [ ]:
C_vals     = [0.1, 0.5, 1.0, 1.414, 2.0, 3.0]
wr_red_C   = []
wr_yel_C   = []

print('Experimento 5 — Barrido de C (200 sims)...')
for C in C_vals:
    r = evaluate(Cristian, RandomAgent, N, -1, agent_kw={'simulations': 200, 'C': C})
    y = evaluate(Cristian, RandomAgent, N,  1, agent_kw={'simulations': 200, 'C': C})
    wr_r = round(r['wr'] * 100, 1)
    wr_y = round(y['wr'] * 100, 1)
    wr_red_C.append(r['wr'])
    wr_yel_C.append(y['wr'])
    print('  C=' + str(C).ljust(5) + ': Rojo=' + str(wr_r) + '%  Amarillo=' + str(wr_y) + '%')

print('\nExperimento 5 completado.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(C_vals, [v * 100 for v in wr_red_C],
        'o-', color='#e74c3c', lw=2.5, ms=9, label='Como Rojo')
ax.plot(C_vals, [v * 100 for v in wr_yel_C],
        's--', color='#e67e22', lw=2.5, ms=9, label='Como Amarillo')
ax.axhline(50, color='#7f8c8d', ls=':', lw=1.5, label='50 % minimo')
ax.axvline(1.414, color='#9b59b6', ls='--', lw=1.8, alpha=0.7,
           label='C = sqrt(2) ~ 1.414  (valor estandar)')

ax.set_xlabel('Constante de exploracion C  (UCB1)')
ax.set_ylabel('Tasa de victoria (%)')
ax.set_title('Exp. 5 — Win rate vs Constante C  |  200 sims, N=' + str(N),
             fontweight='bold')
ax.set_ylim(0, 110)
ax.legend()
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig('groups/Cristian/exp5_C.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafica guardada: exp5_C.png')

---
## Resumen visual — Win rate de todas las configuraciones

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel izquierdo: win-rate como Rojo en Exp 2 y 3
ax = axes[0]
configs = ['sims=25', 'sims=50', 'sims=100', 'sims=200', 'sims=500']
ax.barh(configs, [v * 100 for v in wr_red_sim], color='#e74c3c', alpha=0.85)
ax.axvline(50, color='gray', ls='--', lw=1)
ax.set_xlabel('Win rate (%)')
ax.set_xlim(0, 110)
ax.set_title('Exp. 2: Simulaciones  |  Como Rojo', fontweight='bold')
for i, v in enumerate([v * 100 for v in wr_red_sim]):
    ax.text(v + 1, i, str(round(v, 1)) + '%', va='center', fontsize=9)

# Panel derecho: win-rate como Amarillo en Exp 2 y 3
ax = axes[1]
ax.barh(configs, [v * 100 for v in wr_yel_sim], color='#e67e22', alpha=0.85)
ax.axvline(50, color='gray', ls='--', lw=1)
ax.set_xlabel('Win rate (%)')
ax.set_xlim(0, 110)
ax.set_title('Exp. 2: Simulaciones  |  Como Amarillo', fontweight='bold')
for i, v in enumerate([v * 100 for v in wr_yel_sim]):
    ax.text(v + 1, i, str(round(v, 1)) + '%', va='center', fontsize=9)

fig.suptitle('Resumen — Win rate vs Numero de simulaciones', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('groups/Cristian/resumen_winrate.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafica guardada: resumen_winrate.png')

---
## Análisis de Resultados y Conclusiones

### Experimento 1 — Rendimiento base
El agente MCTS **nunca pierde** contra el jugador aleatorio y supera el 50 % de victorias en ambos colores,
cumpliendo los dos requisitos mínimos de la rúbrica.
La ventaja como Rojo (primer jugador) es consistente con la teoría: Connect-4 está resuelto y Rojo gana con juego perfecto.

### Experimento 2 — Número de simulaciones
Con tan solo 25–50 simulaciones el agente ya supera al aleatorio de forma consistente.
El win-rate **satura** en torno a 200–500 simulaciones contra el aleatorio:
gastar más recursos no produce mejora proporcional frente a un oponente tan débil,
pero sí sería relevante contra agentes más fuertes.
**Cuello de botella identificado:** la función `_winner()` recorre todo el tablero O(ROWS×COLS) = 168 iteraciones
en cada movimiento del rollout, lo que limita cuántas simulaciones se pueden hacer en tiempo acotado.

### Experimento 3 — Dos versiones
La versión con acción táctica inmediata (V1) supera consistentemente a la versión pura MCTS (V2),
especialmente en posiciones donde existe un movimiento ganador o bloqueante.
Con pocas simulaciones (200), el árbol MCTS a veces no alcanza a "ver" la jugada inmediata;
el módulo táctico garantiza que estas posiciones se resuelvan correctamente en O(cols) sin costo adicional.
Esto evidencia que **heurística de dominio + búsqueda** supera a búsqueda pura con profundidad limitada.

### Experimento 4 — Self-play
El agente como Rojo (primer jugador) tiende a ganar más que como Amarillo,
reflejo de la ventaja estructural del primer jugador en Connect-4.
El sesgo se reduce pero no desaparece porque el MCTS no converge al juego perfecto con 200 simulaciones.

### Experimento 5 — Constante C
Valores muy bajos de C (< 0.5) generan explotación excesiva: el árbol converge a la primera rama prometedora
sin explorar alternativas, produciendo alta varianza.
Valores muy altos (> 2.5) diluyen la exploración hasta aproximar el comportamiento aleatorio.
El valor teórico C = √2 ≈ 1.414 es robusto en ambos colores, validando la elección por defecto del agente.

---
## Propuesta de Mejoras

### Debilidad 1 — Selección adversarial incorrecta *(impacto alto)*

**Cuello de botella:** En `_select`, el código siempre maximiza UCB1 desde la perspectiva de `me`,
*incluso en los nodos del oponente*.
Esto modela al oponente como si quisiera que el agente ganara (cooperativo en vez de adversarial).
Con muchas simulaciones y contra el aleatorio el efecto se mitiga, pero contra un agente fuerte
el árbol subestima las amenazas del oponente en posiciones de media partida.

**Propuesta (negamax UCT):** Invertir la recompensa en cada nivel del backpropagation:
```python
def _backpropagate(self, node, reward):
    while node is not None:
        node.visits += 1
        node.wins   += reward
        reward       = 1.0 - reward   # <- invertir perspectiva en cada nivel
        node = node.parent
```
**Causa-efecto:** El árbol modelaría correctamente el juego adversarial → mejor rendimiento
contra oponentes fuertes sin costo computacional extra.

### Debilidad 2 — `_winner()` O(ROWS×COLS) en cada rollout *(impacto medio)*

**Cuello de botella:** La verificación de victoria recorre las 168 celdas del tablero
después de *cada* ficha colocada durante el rollout.
Con 500 simulaciones × 42 movimientos máx. ≈ 3.5 M operaciones por movimiento del agente.
El Experimento 2 muestra que el rendimiento satura a 200–500 sims porque el tiempo
de cómputo lo impide más que la calidad del árbol.

**Propuesta:** Verificador incremental que solo revisa las 4 direcciones alrededor
de la última ficha colocada (máx. 15 celdas en vez de 168 → ~11× más rápido).
Esto permitiría pasar de 500 a ~5000 simulaciones en el mismo tiempo,
mejorando la calidad de decisión especialmente en posiciones complejas de media partida.

### Debilidad 3 — Rollouts puramente aleatorios *(impacto medio-alto)*

**Cuello de botella:** La fase de simulación usa jugadas uniformemente aleatorias,
ignorando completamente el dominio del juego.
Esto requiere muchos rollouts para estimar correctamente el valor de posiciones complejas.

**Propuesta:** Rollouts guiados por heurística ligera:
con probabilidad `p` jugar la mejor columna según una evaluación rápida
(columna central, completar 3 en línea, bloquear 3 del oponente);
con probabilidad `1-p` jugar aleatoriamente.
Incluso `p = 0.3` reduce significativamente la varianza de las estimaciones,
permitiendo que el árbol converja más rápido con el mismo presupuesto de simulaciones.